# 7.7 Ethics and Responsible AI

Fairness metrics (demographic parity, equalized odds, calibration), disparate impact analysis, and model card template.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(42)

n = 2000
group = rng.choice([0,1], n)
credit = rng.normal(650+group*30, 80, n).clip(300,850)
y_true = (credit >= 620).astype(int)
bias = np.where(group==1, -25, 0)
y_pred = (credit+bias >= 620).astype(int)

rate_a, rate_b = y_pred[group==0].mean(), y_pred[group==1].mean()
dp_gap = abs(rate_a - rate_b)
di = min(rate_a,rate_b)/max(rate_a,rate_b)

print(f'Group A approval: {rate_a:.3f}')
print(f'Group B approval: {rate_b:.3f}')
print(f'Demographic Parity gap: {dp_gap:.3f}')
print(f'Disparate Impact ratio: {di:.3f}  (threshold 0.8)')

In [ ]:
def tpr_fpr(g):
    mask=group==g; yt,yp=y_true[mask],y_pred[mask]
    tp=((yp==1)&(yt==1)).sum(); fp=((yp==1)&(yt==0)).sum()
    tn=((yp==0)&(yt==0)).sum(); fn=((yp==0)&(yt==1)).sum()
    return tp/(tp+fn+1e-10), fp/(fp+tn+1e-10)

tpr_a,fpr_a = tpr_fpr(0); tpr_b,fpr_b = tpr_fpr(1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(['Group A','Group B'],[rate_a,rate_b],color=['steelblue','tomato'],alpha=0.8)
axes[0].set(ylabel='Approval Rate',title=f'Demographic Parity (gap={dp_gap:.3f})')
axes[0].set_ylim(0,1); axes[0].grid(True,axis='y',alpha=0.3)

x=np.arange(2)
axes[1].bar(x-0.2,[tpr_a,fpr_a],0.4,label='Group A',color='steelblue',alpha=0.8)
axes[1].bar(x+0.2,[tpr_b,fpr_b],0.4,label='Group B',color='tomato',alpha=0.8)
axes[1].set(xticks=x,xticklabels=['TPR','FPR'],title='Equalized Odds')
axes[1].legend(); axes[1].grid(True,axis='y',alpha=0.3)

bins=np.linspace(300,850,50); bc=(bins[:-1]+bins[1:])/2
axes[2].plot(bc,np.histogram(credit[group==0],bins)[0],lw=2,label='Group A')
axes[2].plot(bc,np.histogram(credit[group==1],bins)[0],lw=2,linestyle='--',label='Group B')
axes[2].axvline(620,color='k',linestyle=':',label='Threshold')
axes[2].set(xlabel='Credit Score',ylabel='Count',title='Score Distributions')
axes[2].legend(); axes[2].grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig('output/responsible_ai.png')
print('Saved responsible_ai.png')